# VizEx — validation analysis

Do the features recovered from a participant's recreation track the features of
the target image they were shown? Validation trials only (`stimulus_type == IMAGE`,
condition codes A00–C14).

Two models per feature:
1. **Continuous** — `recreation ~ target`, random intercept per participant.
   The slope answers whether recreations track targets.
2. **Categorical** — `recreation ~ C(condition_code)`, random intercept per
   participant. The omnibus Wald test answers whether target identity matters at all.

Requires `015_feature_extraction` to have been run first.

In [ ]:
## imports

import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

In [ ]:
## paths

PROJECT_PATH = Path().resolve().parent
TABLES_PATH = PROJECT_PATH / "015_tables"

TRIALS_FEATURES_CSV = TABLES_PATH / "vizex_features.csv"
TARGETS_FEATURES_CSV = TABLES_PATH / "vizex_targets_features.csv"

FEATURES = [
    ("detail", "Normalised detail level"),
    ("r_directionality", "Normalised radial orientation"),
    ("theta_directionality", "Normalised tangential orientation"),
    ("brightness", "Mean image brightness (0-1)"),
    ("contrast", "Mean image contrast (RMS)"),
]

TEXT_SIZE = 15
TITLE_SIZE = int(TEXT_SIZE * 1.15)

In [ ]:
## load the feature tables

df_trials = pd.read_csv(TRIALS_FEATURES_CSV)
df_targets = pd.read_csv(TARGETS_FEATURES_CSV)

df_trials["condition_code"] = df_trials["condition_code"].astype(str)
df_targets["condition_code"] = df_targets["condition_code"].astype(str)

# Guard against a stale or half-written table from 015.
uncropped = df_trials["png_filename"].str.contains("uncropped", na=False)
if uncropped.any():
    raise ValueError(f"{int(uncropped.sum())} uncropped rows present -- re-run 015.")

for name, d in (("trials", df_trials), ("targets", df_targets)):
    for feature, _ in FEATURES:
        if feature not in d.columns:
            raise KeyError(f"[{name}] missing feature column '{feature}' -- re-run 015.")
        if d[feature].isna().all():
            raise ValueError(f"[{name}] feature column '{feature}' is all NaN -- re-run 015.")

print(f"trials:  {len(df_trials):,} rows, {df_trials['participant_code'].nunique()} participants")
print(f"targets: {len(df_targets):,} rows")

display(df_trials.head())
display(df_targets.head())

In [ ]:
## helpers

def is_validation_code(x) -> bool:
    """True for target image codes like A00, B07, C14."""
    return bool(re.fullmatch(r"[ABC]\d{2}", str(x).strip()))


def validation_sort_key(code):
    """Order A00..A14, B00..B14, C00..C14."""
    m = re.fullmatch(r"([A-Z])(\d+)", str(code).strip())
    return (ord(m.group(1)), int(m.group(2)), str(code)) if m else (999, 999, str(code))


def model_summary_string(mdf, var, precision=3) -> str:
    b, se, p = mdf.params[var], mdf.bse[var], mdf.pvalues[var]
    if p < 0.0001:
        p_str = "p < .0001"
    elif p < 0.001:
        p_str = "p < .001"
    else:
        p_str = f"p = {p:.3f}".replace("0.", ".")
    return f"b = {b:.{precision}f}, SE = {se:.{precision}f}, {p_str}"


def prepare_validation_df(feature: str) -> pd.DataFrame:
    """Long frame: condition_code, participant_code, <feature>_rec, <feature>_targ."""
    rec = df_trials[df_trials["stimulus_type"].astype(str).str.upper().eq("IMAGE")]
    rec = rec[rec["condition_code"].map(is_validation_code)]
    targ = df_targets[df_targets["condition_code"].map(is_validation_code)]

    df = pd.merge(
        rec[["condition_code", "participant_code", feature]].rename(
            columns={feature: f"{feature}_rec"}),
        targ[["condition_code", feature]].rename(columns={feature: f"{feature}_targ"}),
        on="condition_code", how="left",
    ).dropna(subset=[f"{feature}_rec", f"{feature}_targ"])

    df["cond_order"] = df["condition_code"].map(validation_sort_key)
    return (df.sort_values(["cond_order", "participant_code"])
              .drop(columns=["cond_order"])
              .reset_index(drop=True))


def fit_mixedlm(formula: str, df: pd.DataFrame):
    """Fit with a random participant intercept, trying optimisers until one converges."""
    md = smf.mixedlm(formula, df, groups=df["participant_code"])
    last = None
    for method in ("lbfgs", "bfgs", "cg", "powell", "nm"):
        try:
            last = md.fit(reml=False, method=method, maxiter=2000, disp=False)
            if getattr(last, "converged", False):
                break
        except Exception:
            continue
    if last is None:
        raise RuntimeError(f"MixedLM fit failed for all optimizers: {formula}")
    return md, last


def variance_components(md, mdf):
    """Signal (fixed-effect), DT (participant), DN (residual), plus a singularity flag.

    statsmodels reports 'Random effects covariance is singular' on several of these
    fits. When the participant variance collapses to ~0 the DT estimate is not
    identified, so it is flagged rather than reported silently.
    """
    signal_var = float(np.var(md.predict(mdf.params)))
    dt_var = float(mdf.cov_re.iloc[0, 0])
    dn_var = float(mdf.scale)
    singular = bool(not np.isfinite(dt_var) or dt_var < 1e-10)
    return signal_var, dt_var, dn_var, singular

In [ ]:
## plots

def plot_feature_categorical(feature, df, *, y_label, jitter=0.12, seed=0):
    """Target marked with an X per validation trial; recreations jittered around it."""
    rng = np.random.RandomState(seed)
    codes = sorted(df["condition_code"].unique(), key=validation_sort_key)
    code_to_x = {c: i for i, c in enumerate(codes)}
    xs = df["condition_code"].map(code_to_x).to_numpy(dtype=float)

    fig, ax = plt.subplots(figsize=(max(10, len(codes) * 0.35), 4.6))
    ax.scatter(xs + rng.normal(0, jitter, size=len(xs)),
               df[f"{feature}_rec"].to_numpy(float), s=14, alpha=0.55)

    targ = df.groupby("condition_code", sort=False)[f"{feature}_targ"].first()
    ax.scatter([code_to_x[c] for c in targ.index], targ.to_numpy(float), s=70, marker="x")

    ax.set_title(f"{feature} validation (categorical)", fontsize=TITLE_SIZE)
    ax.set_ylabel(y_label, fontsize=TEXT_SIZE)
    ax.set_xlabel("validation trial", fontsize=TEXT_SIZE)
    ax.set_xticks(range(len(codes)))
    ax.set_xticklabels(codes, rotation=90, fontsize=9)
    ax.tick_params(axis="y", labelsize=TEXT_SIZE)
    ax.grid(False)
    plt.tight_layout()
    plt.show()


def plot_feature_continuous(feature, df, *, y_label):
    """Recreation against target, with an OLS fit and the y = x ideal."""
    rec_col, targ_col = f"{feature}_rec", f"{feature}_targ"
    x = df[targ_col].to_numpy(float)
    y = df[rec_col].to_numpy(float)

    fig, ax = plt.subplots(figsize=(6.4, 5.2))
    ax.scatter(x, y, s=14, alpha=0.55)

    m = smf.ols(f"{rec_col} ~ {targ_col}", data=df).fit()
    xx = np.linspace(np.nanmin(x), np.nanmax(x), 200)
    ax.plot(xx, m.params["Intercept"] + m.params[targ_col] * xx, linewidth=2)
    ax.plot([x.min(), x.max()], [x.min(), x.max()], linestyle=":", linewidth=2)

    ax.set_title(f"{feature} validation (continuous)", fontsize=TITLE_SIZE)
    ax.set_xlabel(f"target {feature}", fontsize=TEXT_SIZE)
    ax.set_ylabel(f"recreated {feature}", fontsize=TEXT_SIZE)
    ax.tick_params(axis="both", labelsize=TEXT_SIZE)
    plt.tight_layout()
    plt.show()

## Continuous model: recreation ~ target

In [ ]:
results, validation_dfs = [], {}

for feature, y_label in FEATURES:
    df = prepare_validation_df(feature)
    validation_dfs[feature] = df

    md, mdf = fit_mixedlm(f"{feature}_rec ~ {feature}_targ", df)
    targ_col = f"{feature}_targ"
    signal_var, dt_var, dn_var, singular = variance_components(md, mdf)

    if singular:
        print(f"  WARNING [{feature}]: random-effects variance ~0 (DT = {dt_var:.3g}); "
              f"treat DT as unidentified.")

    results.append({
        "Feature": feature,
        "b": float(mdf.params.get(targ_col, np.nan)),
        "se": float(mdf.bse.get(targ_col, np.nan)),
        "z": float(mdf.tvalues.get(targ_col, np.nan)),
        "p": float(mdf.pvalues.get(targ_col, np.nan)),
        "n": len(df),
        "Signal_Val": signal_var,
        "DT": dt_var,
        "DN": dn_var,
        "RE_singular": singular,
        "converged": bool(getattr(mdf, "converged", False)),
    })

    plot_feature_categorical(feature, df, y_label=y_label)
    plot_feature_continuous(feature, df, y_label=y_label)

validation_results = pd.DataFrame(results)
validation_results.to_csv(TABLES_PATH / "validation_results_full.csv", index=False)
display(validation_results)

## Categorical model: recreation ~ C(condition_code)

In [ ]:
results_cat = []

for feature, y_label in FEATURES:
    df = validation_dfs[feature]

    md, mdf = fit_mixedlm(f"{feature}_rec ~ C(condition_code)", df)
    signal_var, dt_var, dn_var, singular = variance_components(md, mdf)

    try:
        table = mdf.wald_test_terms().table
        stat = float(table.loc["C(condition_code)", "statistic"])
        p = float(table.loc["C(condition_code)", "pvalue"])
    except Exception:
        stat, p = np.nan, np.nan

    if singular:
        print(f"  WARNING [{feature}]: random-effects variance ~0 (DT = {dt_var:.3g}); "
              f"treat DT as unidentified.")

    results_cat.append({
        "Feature": feature,
        "Wald_Stat": stat,
        "p": p,
        "n": len(df),
        "Signal_Val": signal_var,
        "DT": dt_var,
        "DN": dn_var,
        "RE_singular": singular,
        "converged": bool(getattr(mdf, "converged", False)),
    })

    plot_feature_categorical(feature, df, y_label=y_label)

categorical_results = pd.DataFrame(results_cat)
categorical_results.to_csv(TABLES_PATH / "validation_results_categorical.csv", index=False)
display(categorical_results)